In [ ]:
import os
import joblib
import optuna
import difflib
import logging
import pandas as pd
import lightgbm as lgb
from typing import Optional, List
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report
from sklearn.preprocessing import StandardScaler
import numpy as np

# === Logging ===
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

# === Paths ===
DATA_PATH = r"C:\Users\Kmal Elashry\Downloads\Cleaned_TOI.xlsx"  # Update with your dataset path
SAVE_DIR = r"C:\Records\KAMAL\Nasa"
os.makedirs(SAVE_DIR, exist_ok=True)
MODEL_PATH = os.path.join(SAVE_DIR, "exoplanet_model_userfriendly_optuna.joblib")

# === Feature mapping ===
FEATURE_MAPPING = {
    "transit_depth": ["koi_depth", "koi_dprad", "transit_depth", "depth", "pl_trandep"],
    "transit_duration": ["koi_duration", "transit_duration", "pl_trandurh", "duration"],
    "transit_epoch": ["koi_time0bk", "transit_epoch", "epoch", "pl_tranmid", "t0"],
    "orbital_period": ["koi_period", "orbital_period", "period", "pl_orbper", "orb"],
    "planet_to_star_radius_ratio": ["pl_ratro", "koi_ror", "planet_to_star_radius_ratio", "ror", "radius_ratio"],
    "planet_radius": ["pl_rade", "pl_radj", "pl_radii", "radius", "planet_diameter", "planet radius", "koi_prad", "rp", "planet_rad", "planet_radius"],
    "stellar_radius": ["koi_srad", "st_rad", "stellar_radius", "star_radius"],
    "temperature": ["pl_eqt", "pl_eqt_err1", "pl_eqt_err2", "koi_teq", "eq_temp", "equilibrium_temperature", "planet_temperature", "temperature", "temp"],
    "stellar_temperature": ["st_teff", "koi_steff", "stellar_teff", "effective_temperature", "star_temp"],
    "distance": ["sy_dist", "st_dist", "distance", "distance_from_earth", "dist_from_earth", "dist", "d_planet", "star_distance", "sys_dist", "koi_dor"]
}
DEFAULT_VALUES= {
           'transit_depth': 4376.0
           , 'transit_duration': 2.750763
           , 'transit_epoch': 2459632.099977
           , 'orbital_period': 4.244335
           , 'planet_radius': 10.5416
           , 'stellar_radius': 1.23975
           , 'temperature': 1201.5124066
           , 'stellar_temperature': 5805.6
           ,'distance': 386.525
           , 
           }

# === Load data ===
def load_data(path: str) -> pd.DataFrame:
    if path.endswith(".csv"):
        return pd.read_csv(path, comment="#", engine="python")
    elif path.endswith((".xls", ".xlsx")):
        raw = pd.read_excel(path, header=None, dtype=str)
        header_row = None
        for i, row in raw.iterrows():
            if row.dropna().str.contains(r"(pl_|koi_|st_)", case=False, regex=True).any():
                header_row = i
                break
        if header_row is None:
            raise ValueError("Could not detect a valid header row in Excel file")
        return pd.read_excel(path, skiprows=header_row)
    else:
        raise ValueError("Unsupported file type: " + path)

# === Detect target ===
def detect_target(df: pd.DataFrame, target_candidates: List[str]) -> Optional[str]:
    lower_cols = {c.lower(): c for c in df.columns}
    for cand in target_candidates:  # exact
        if cand.lower() in lower_cols:
            return lower_cols[cand.lower()]
    for col in df.columns:  # substring
        col_lower = col.lower()
        for cand in target_candidates:
            if cand.lower() in col_lower or col_lower in cand.lower():
                return col
    for cand in target_candidates:  # fuzzy
        matches = difflib.get_close_matches(cand.lower(), lower_cols.keys(), n=1, cutoff=0.6)
        if matches:
            return lower_cols[matches[0]]
    for col in df.columns:  # keyword fallback
        low = col.lower()
        if any(k in low for k in ["dispos", "status", "label", "confirm", "fp", "planet", "archive"]):
            return col
    return None

# === Resolve features ===
def resolve_features(df: pd.DataFrame, feature_mapping: dict):
    resolved, missing, used_columns = {}, [], set()
    lower_cols = {col.lower(): col for col in df.columns}
    for user_feat, candidates in feature_mapping.items():
        found = False
        for cand in candidates:
            cand_lower = cand.lower()
            if cand_lower in lower_cols:
                col_name = lower_cols[cand_lower]
                if col_name not in used_columns:
                    resolved[user_feat] = col_name
                    used_columns.add(col_name)
                    found = True
                    break
            for col_lower, col_orig in lower_cols.items():  # fuzzy
                if cand_lower in col_lower or col_lower in cand_lower:
                    if col_orig not in used_columns:
                        resolved[user_feat] = col_orig
                        used_columns.add(col_orig)
                        found = True
                        break
            if found:
                break
        if not found:
            missing.append(user_feat)
    return resolved, missing
TARGET_CANDIDATES = [
    "planet disposition", "archive disposition", "koi_disposition",
    "disposition", "disposition_flag", "pl_disposition",
    "tfopwg_dispostion", "dispostion","tfopwg_disp"
]

# === Train & Save Model ===
def train_and_save_model():
    df = load_data(DATA_PATH)

    # --- Detect target column ---
    target_col = detect_target(df,TARGET_CANDIDATES )
    if target_col is None:
        raise ValueError("Target/disposition column not found.")

    # --- Resolve features ---
    resolved, missing = resolve_features(df, FEATURE_MAPPING)
    logger.info("Resolved features: %s", resolved)
    if missing:
        logger.warning("Missing features: %s", missing)

    # --- Prepare data ---
    y = df[target_col].astype(str)  # use raw labels directly
    X = df[list(resolved.values())].copy()
    X = X.apply(pd.to_numeric, errors="coerce").fillna(X.median(numeric_only=True))

    print("Unique classes in target (raw):", y.unique())

    # Drop rows with missing target
    mask = y.notna()
    X, y = X.loc[mask], y.loc[mask]

    logger.info("Class distribution: %s", y.value_counts().to_dict())
    if y.nunique() < 2:
        raise ValueError(f"Need at least 2 classes to train. Found: {y.unique().tolist()}")

    # --- Normalize features ---
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)

    X_train, X_test, y_train, y_test = train_test_split(
        X_scaled, y, test_size=0.2, stratify=y, random_state=42
    )

    def objective(trial):
        params = {
        "boosting_type": "gbdt",
        "num_leaves": trial.suggest_int("num_leaves", 16, 512),
        "learning_rate": trial.suggest_float("learning_rate", 1e-4, 0.3, log=True),
        "n_estimators": trial.suggest_int("n_estimators", 200, 2000),
        "max_depth": trial.suggest_int("max_depth", -1, 16),
        "feature_fraction": trial.suggest_float("feature_fraction", 0.5, 1.0),
        "bagging_fraction": trial.suggest_float("bagging_fraction", 0.5, 1.0),
        "bagging_freq": trial.suggest_int("bagging_freq", 1, 10),
        "min_child_samples": trial.suggest_int("min_child_samples", 5, 100),
        "lambda_l1": trial.suggest_float("lambda_l1", 1e-4, 10.0, log=True),
        "lambda_l2": trial.suggest_float("lambda_l2", 1e-4, 10.0, log=True),
        "seed": 42
            }


        n_classes = len(np.unique(y_train))
        if n_classes == 2:
            params["objective"] = "binary"
            params["metric"] = "binary_logloss"
        else:
            params["objective"] = "multiclass"
            params["metric"] = "multi_logloss"
            params["num_class"] = n_classes

        model = lgb.LGBMClassifier(**params)
        model.fit(X_train, y_train)
        preds = model.predict(X_test)
        return accuracy_score(y_test, preds)

    # --- Run Optuna search ---
    study = optuna.create_study(direction="maximize")
    study.optimize(objective, n_trials=30)

    best_params = study.best_trial.params
    n_classes = len(np.unique(y_train))
    if n_classes > 2:
        best_params["objective"] = "multiclass"
        best_params["num_class"] = n_classes
    else:
        best_params["objective"] = "binary"

    # --- Train best model ---
    best_model = lgb.LGBMClassifier(**best_params)
    best_model.fit(X_train, y_train)

    preds = best_model.predict(X_test)
    print("\n=== Model Evaluation ===")
    print("Accuracy:", accuracy_score(y_test, preds))
    print(classification_report(y_test, preds))

    # Save model + scaler + features
    joblib.dump({
    "model": best_model,
    "scaler": scaler,
    "features": resolved,
    "target": target_col,
    "defaults": DEFAULT_VALUES
}, MODEL_PATH)
    print(f"Model saved at {MODEL_PATH}")

# === Prediction for Backend (with normalization) ===
def predict_new_data(new_input: dict):
    saved = joblib.load(MODEL_PATH)
    model, scaler, features, target_col , defaults = saved["model"], saved["scaler"], saved["features"], saved["target"] , saved["defaults"]

    # Map user-friendly keys -> dataset feature names
    mapped_input = {}
    for user_key, col_name in features.items():
        if user_key in new_input:
            mapped_input[col_name] = new_input[user_key]
        else:
            dval = defaults.get(user_key, 0)  # fallback
            mapped_input[col_name] = dval
    new_df = pd.DataFrame([mapped_input])
    X_new = new_df.reindex(columns=list(features.values()), fill_value=0)
    X_new = X_new.apply(pd.to_numeric, errors="coerce").fillna(X_new.median(numeric_only=True))

    # Apply saved normalization
    X_new_scaled = scaler.transform(X_new)

    # Get probability predictions
    probs = model.predict_proba(X_new_scaled)[0]
    probs_dict = {cls: float(round(p * 100, 2)) for cls, p in zip(model.classes_, probs)}

    # Aggregate into "exo" vs "not"
    exo_prob = probs_dict.get("CONFIRMED", 0) + probs_dict.get("CANDIDATE", 0)
    non_exo_prob = probs_dict.get("FALSE POSITIVE", 0)

    return {
        "Exoplanet Probability": round(exo_prob, 2),
        "Non-Exoplanet Probability": round(non_exo_prob, 2),
        "Details": probs_dict
    }


def load_model():
    if not os.path.exists(MODEL_PATH):
        raise FileNotFoundError("Model file not found! Run training first.")
    saved = joblib.load(MODEL_PATH)
    return saved["model"], saved["scaler"], saved["features"], saved["target"] , saved["defaults"]




In [ ]:
# === Example usage ===
if __name__ == "__main__":
    # Train and save the model
    train_and_save_model()

    # Reload trained model + features
    model, scaler, features, target, defaults = load_model()

    # Example input
    example_input = pd.read_csv(r"C:\Users\Kmal Elashry\Downloads\test.csv")
# Detect missing features
    provided_keys = set(example_input.keys())
    expected_keys = set(features.keys())
    missing_keys = expected_keys - provided_keys

    if missing_keys:
        print("\n=== Missing Features Filled with Defaults ===")
        for key in missing_keys:
            print(f"{key}: {defaults.get(key, 0)}")

    result = predict_new_data(example_input)
    logger.info("Prediction result: %s", result)
    print(f'Prediction result: {result}')


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix
from sklearn.preprocessing import LabelEncoder

# Load the dataframe
df = load_data(DATA_PATH)
df2= df.copy()
for col in df.select_dtypes(include="object").columns:
    le = LabelEncoder()
    df[col] = le.fit_transform(df[col].astype(str))

# Optional: Visualize feature correlations
numeric_df = df.select_dtypes(include=[np.number])
full_corr_matrix = numeric_df.corr()

if target in full_corr_matrix.columns:
    target_correlation = full_corr_matrix[target].sort_values(ascending=False)
    print("\nCorrelation with Target (Planet Disposition):")
    print(target_correlation)

    plt.figure(figsize=(8, 12))
    sns.heatmap(target_correlation.to_frame(), annot=True, cmap='coolwarm')
    plt.title("Feature Correlation with Planet Disposition")
    plt.show()
else:
    print(f"\nTarget column '{target}' is not numeric and cannot be used for correlation.")

# Feature importance
fi = model.feature_importances_
idx = np.argsort(fi)[::-1][:30]
plt.figure(figsize=(10,6))
plt.barh(range(len(idx)), fi[idx][::-1])
plt.yticks(range(len(idx)), [list(features.values())[i] for i in idx][::-1])
plt.title('Top Feature Importances')
plt.tight_layout()
plt.show()

# Display the confusion matrix
print("\n✅ Confusion Matrix:")

# Prepare features and target for confusion matrix
resolved, _ = resolve_features(df2, FEATURE_MAPPING)
target_col = detect_target(df2, TARGET_CANDIDATES)
y = df2[target_col].astype(str)
X = df2[list(resolved.values())].copy()
X = X.apply(pd.to_numeric, errors="coerce").fillna(X.median(numeric_only=True))

# Drop rows with missing target
mask = y.notna()
X, y = X.loc[mask], y.loc[mask]

# Normalize features
X_scaled = scaler.transform(X)

from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.2, stratify=y, random_state=42
)

y_pred_test = model.predict(X_test)

# Use actual unique labels from y_test for confusion matrix
unique_labels = sorted(y_test.unique())
cm = confusion_matrix(y_test, y_pred_test, labels=unique_labels)
sns.heatmap(cm, annot=True, fmt='d',
            xticklabels=unique_labels,
            yticklabels=unique_labels)
plt.ylabel('Actual')
plt.xlabel('Predicted')
plt.title('Confusion Matrix')
plt.show()